## Notebook Description: Data Curation and Averaging of Gas and Nutrition Data

This notebook outlines a comprehensive data curation and preprocessing workflow for two datasets: 'gas' and 'nutrition'. The primary objectives include standardizing categorical variables, filtering out erroneous or incomplete records, and aggregating replicate measurements. Key methodologies involve leveraging the pandas library for data manipulation, including the use of `df.replace()` for consistent categorization (e.g., 'Breding ' to 'Breeding'), `df.str.contains()` and `df.isna()` for identifying and marking rows for deletion based on specific keywords in remark columns or missing 'methane_intensity' values, and `df.groupby().mean()` for calculating average values and replicate counts (e.g., $\text{n_replicates_gas}$, $\text{n_replicates_nutrition}$) for each unique sample identifier ($\text{id_lab}$). The outcome is a set of cleaned and averaged datasets ready for further analysis, specifically $\text{gas_clean_av2}$ and $\text{nutrition_av2}$, which condense experimental replicates into single, representative entries.

# 1.0 Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd

# 2.0 Data Import

In [3]:
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_gas_sorted_by_sql.csv')
nutrition = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_nutrition_sorted_by_sql.csv')

In [4]:
gas.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
0,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,33.240646,0,95.43754089,34.829738,Standar,NaN,NaN,NaN,NaN,NaN
1,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,32.485469,0,89.75200777,36.194699,Standar,NaN,NaN,NaN,NaN,NaN


In [5]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2'],
      dtype='object')

In [6]:
nutrition.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


# 3.0 Formatting

In [7]:
gas.requisitioner.unique()

array(['LMF', 'Genetic bank', 'Breeding', nan, 'Set 2', 'Breding ',
       'LMF-invivo', 'Benchmark', 'Genbank', 'Isabel Molina',
       'Mauricio Sotelo', 'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [8]:
# Define the mapping for functional groups
requisitioner_mapping = {
    'Breding ': 'Breeding',
    'Genbank': 'Genetic_bank',
    'Genetic bank': 'Genetic_bank'
}

# Apply the mapping to centers datt
gas['requisitioner'] = gas['requisitioner'].replace(requisitioner_mapping)
nutrition['requisitioner'] = nutrition['requisitioner'].replace(requisitioner_mapping)

print('Updated requisitioner counts for gas:')
display(gas['requisitioner'].value_counts())

print('Updated requisitioner counts for nutrition:')
display(nutrition['requisitioner'].value_counts())

Updated requisitioner counts for gas:


,count
requisitioner,
Genetic_bank,4235
Breeding,1725
LMF,1631
Benchmark,150
LMF-invivo,75
Jacobo Arango/ Alejandro Montoya,35
Jacobo Arango/Alejandro Montoya,35
Mauricio Sotelo,8
Isabel Molina,8


Updated requisitioner counts for nutrition:


,count
requisitioner,
Genetic_bank,1014
LMF,434
Breeding,370
Jacobo Arango/ Alejandro Montoya,30
Benchmark,28
Jacobo Arango/Alejandro Montoya,20
LMF-invivo,14
Mauricio Sotelo,2
Isabel Molina,2


# 4.0 Curation

In [9]:
remarks_columns = ['information_remarks_1', 'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2']

# Convert columns to string type and check for 'elim' or 'descar' (case-insensitive)
conditions = []
for col in remarks_columns:
    contains_elim = gas[col].astype(str).str.contains('elim', na=False, case=False)
    contains_descar = gas[col].astype(str).str.contains('descar', na=False, case=False)
    conditions.append(contains_elim | contains_descar)

# Combine the remarks conditions using OR logic (any() along axis=1)
# Create a boolean series where True means 'elim' or 'descar' is found in at least one remark column
has_keywords = pd.concat(conditions, axis=1).any(axis=1)

# New condition: check if 'methane_intensity' is NaN
is_methane_intensity_null = gas['methane_intensity'].isna()

# Combine all conditions for deletion using OR logic
final_delete_condition = has_keywords | is_methane_intensity_null

# Create the 'delete' column, assigning 'yes' or 'no' based on the final condition
gas['delete'] = np.where(final_delete_condition, 'yes', 'no')

In [10]:
gas.to_csv('gas.csv')

In [11]:
gas_clean = gas[gas['delete'] == 'no']

In [12]:
gas_clean.to_csv('gas_clean.csv')

In [13]:
# # Count the number of 'valid' rows (where 'delete' is 'no') for each id_lab and batch combination
# valid_rows_per_id_lab_batch = gas[gas['delete'] == 'no'].groupby(['id_lab', 'batch']).size()

# # Identify (id_lab, batch) combinations that have less than 2 valid rows.
# # These combinations should be entirely excluded.
# id_lab_batch_to_completely_exclude = valid_rows_per_id_lab_batch[valid_rows_per_id_lab_batch < 2].index

# # Create a multi-index from the gas DataFrame to easily filter
# gas_multi_index = gas.set_index(['id_lab', 'batch'])

# # Step 1: Exclude all rows from the identified (id_lab, batch) combinations
# gas_filtered_by_batch_count = gas_multi_index.drop(id_lab_batch_to_completely_exclude, errors='ignore').reset_index()

# # Step 2: From the remaining data, exclude individual rows where 'delete' is 'yes'
# gas_clean = gas_filtered_by_batch_count[gas_filtered_by_batch_count['delete'] == 'no']

In [14]:
#gas.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_complete_subsets_1234_2026_06_09.csv', index=None)

In [15]:
#gas_clean.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_subsets_1234_2026_06_09.csv', index=None)

In [16]:
#nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_complete_1234_2026_06_09.csv', index=None)

# 5.0 Average values

## 5.1 Gas

In [17]:
gas_clean.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [18]:
#Columns processing
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete']
numeric_columns = [ 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm']

for df in [gas_clean]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

/tmp/ipykernel_7481/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_7481/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype('category')
/tmp/ipykernel_7481/3221164181.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/

In [19]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [20]:
gas_clean_av = mean_with_replicates(gas_clean, category_columns, numeric_columns)

/tmp/ipykernel_7481/1161671181.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_7481/1161671181.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_7481/1161671181.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [21]:
gas_clean_av2 = gas_clean_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'net_gas_8h_ml','net_gas_24h_ml','gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [22]:
gas_clean_av2

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,839,LMF-invivo,Dieta 1_Exp1,NaN,30.67,67.46,149.92,34.90,2.31,14.42,14.69,21.82,71.84,321.88,6
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,1019,LMF-invivo,Dieta 1_Exp2,NaN,28.54,58.34,131.09,35.74,2.56,42.80,28.57,39.10,122.89,546.98,13
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,738,LMF-invivo,Dieta 2_Exp1,NaN,29.60,62.15,139.52,30.16,2.16,13.87,14.36,20.02,71.36,317.83,6
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,1022,LMF-invivo,Dieta 2_Exp2,NaN,23.83,47.68,107.07,21.09,1.95,31.77,23.38,24.94,125.08,556.90,13
4,F24-3416,CIAT-705,1,1,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,29.35,61.94,134.59,74.34,5.52,14.26,17.35,23.36,31.84,149.32,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658,F26-0037,CIAT-18947,4,1048,Genetic_bank,Sesbania sesban,Shrub_Trees,35.23,73.90,154.75,51.99,3.36,13.67,13.76,21.30,41.93,-0.47,7
659,F26-0038,CIAT-19165,4,1051,Genetic_bank,Sesbania keniensis,Shrub_Trees,36.44,76.11,157.76,44.03,2.79,14.66,15.21,24.00,56.24,-8.23,7
660,F26-0039,CIAT-21899,4,1054,Genetic_bank,Sesbania sesban,Shrub_Trees,36.09,68.90,143.71,36.07,2.51,14.06,14.81,21.29,61.07,0.43,7
661,F26-0040,CIAT-23414,4,1057,Genetic_bank,Codariocalyx motorius,Shrub_Trees,29.68,59.26,122.99,26.23,2.13,14.60,15.26,18.76,73.03,-0.58,5


In [ ]:
#gas_clean_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_average_subsets_1234_2026_06_09.csv', index=None)

## 5.2 Nutrition

In [23]:
nutrition.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm'],
      dtype='object')

In [24]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']

for df in [nutrition]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [25]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [26]:
nutrition_av = mean_with_replicates(nutrition, category_columns, numeric_columns)

/tmp/ipykernel_7481/1194620962.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_7481/1194620962.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_7481/1194620962.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


In [27]:
nutrition_av.columns

Index(['id_lab', 'subset', 'no', 'requisitioner', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition'],
      dtype='object')

In [28]:
nutrition_av2 = nutrition_av[['id_lab',	'id', 'subset', 'no',	'requisitioner',	'tax_name',	'functional_group',
                              'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm', 'n_replicates_nutrition']]

In [29]:
nutrition_av2.head()

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,89.00,9.50,89.53,17.68,35.48,71.52,2
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,89.00,9.50,89.53,0.00,0.00,0.00,2
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,89.00,9.10,89.58,13.79,33.91,67.86,2
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,237.0,LMF-invivo,Dieta 2_Exp2,NaN,89.00,9.10,89.58,0.00,0.00,0.00,2
4,F24-3416,CIAT-705,1,1.0,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,91.97,12.68,87.32,33.01,28.05,45.22,2


In [30]:
#nutrition_av2.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_average_1234_2026_06_09.csv', index=None)